# Datathon 2026 — Career Success Score Tahmini (v11)

**Görev:** `career_success_score` (0–100) regresyonu | **Metrik:** MSE (düşük = iyi)

## Versiyon Geçmişi

| Versiyon | OOF MSE | Public MSE | Gap | Açıklama |
|----------|---------|------------|-----|----------|
| v2 | ~85.00 | 86.96 | ~+1.96 | Temel özellikler |
| v4 | 76.94 | 87.54 | +10.60 | OOF iyi, public kötü (overfitting) |
| v9 | 76.60 | 86.70 | +10.10 | Dual-Ridge + Skill Interactions |
| v10 | 76.12 | 86.58 | +10.46 | Triple-Ridge + Target Encoding + Grid Blend |
| **v11** | **~71.50** | **hedef ~80.0x** | **~+8.50** | **Dual-Mode BERTurk + Tabular Ensemble** |

## Mimari Özet

### Neden BERTurk Fine-Tuning?

Önceki versiyonlarda metin özellikleri TF-IDF + Ridge (doğrusal) pipeline ile üretiliyordu.
Türkçe'nin morfolojik yapısı, olumsuzluk ekleri ve cümle içi anlamsal bağlar doğrusal
modellerle tam olarak yakalanamaz. Test setinin hedef değişken varyansı 264.53'tür;
OOF MSE'yi ~80 bandına indirmek için modelin varyansın en az %70'ini açıklaması gerekir.

Çözüm olarak `dbmdz/bert-base-turkish-cased` (BERTurk) modeli doğrudan `career_success_score`
tahmine yönelik fine-tune edilmiş ve çıktısı ağaç modellerine meta-özellik olarak beslenmiştir.

### Dual-Mode Tasarım (GPU / CPU Fallback)

Notebook, çalıştığı ortama göre otomatik olarak iki moddan birini seçer:

- **GPU Modu (Kaggle):** BERTurk fine-tuning çalışır; 5-fold OOF tahminleri ağaç modellerine meta-özellik olarak eklenir.
- **CPU Fallback (Yerel):** BERTurk aşaması atlanır; Triple-Ridge + Smooth Target Encoding + Tabular Ensemble (v10 düzeyi) ile tahmin üretilir.


## 1. Kurulum ve Kütüphane Yüklemeleri

Temel veri işleme (`numpy`, `pandas`), makine öğrenimi (`scikit-learn`, `lightgbm`, `xgboost`, `catboost`)
ve derin öğrenme (`torch`, `transformers`) kütüphaneleri içe aktarılır.

PyTorch veya Transformers yüklü değilse `USE_GPU_BERT = False` olarak ayarlanır;
notebook hata vermeden CPU fallback modunda devam eder.


In [1]:
import numpy as np
import pandas as pd
import warnings
import os, time
warnings.filterwarnings('ignore')

from sklearn.model_selection import KFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.linear_model import Ridge

import lightgbm as lgb
import xgboost as xgb
import catboost as cb

SEED    = 42
N_FOLDS = 5
np.random.seed(SEED)

# PyTorch ve Transformers yüklü ise GPU durumuna göre USE_GPU_BERT ayarlanır.
# Kütüphane bulunamazsa ImportError yakalanır ve CPU fallback moduna geçilir.
try:
    import torch
    import transformers
    from torch.utils.data import Dataset, DataLoader
    from torch.optim import AdamW
    from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
    USE_GPU_BERT = torch.cuda.is_available()
except ImportError:
    USE_GPU_BERT = False

print(f'🤖 BERTurk Fine-Tuning Active: {USE_GPU_BERT}')

🤖 BERTurk Fine-Tuning Active: True


In [2]:
# Kaggle yarışma veri seti yükleniyor.
# Yerel ortamda çalıştırırken bu yolları güncellemeniz gerekir.
train = pd.read_csv('/kaggle/input/competitions/datathon-2026/train.csv')
test  = pd.read_csv('/kaggle/input/competitions/datathon-2026/test_x.csv')
sub   = pd.read_csv('/kaggle/input/competitions/datathon-2026/sample_submission.csv')

print(f'Train: {train.shape} | Test: {test.shape}')

Train: (10000, 47) | Test: (10000, 46)


## 2. Sabitler ve Sütun Tanımları

Hedef değişken, ID sütunu ve metin sütunu sabit olarak tanımlanır.
Özellik grupları dört kategoriye ayrılır:

- `SKILL_COLS`: teknik beceri skorları (9 sütun)
- `SOCIAL_COLS`: sosyal beceri skorları (4 sütun)
- `CAT_COLS`: kategorik değişkenler (target encoding ve label encoding uygulanacak)
- `MISSING_COLS`: eksik veri oranı yüksek olan sütunlar (eksiklik gösterge bayrakları üretilecek)

`NEG_WORDS` ve `POS_WORDS` listeleri, mentor geri bildirim metninden duygu sinyali
çıkarmak için kullanılır. Her liste hem ASCII (akansız) hem de Unicode (Türkçe karakterli)
formları içerir; farklı yazım düzenlerinde oluşturulmuş metinlerin yakalanmasını sağlar.


In [3]:
TARGET   = 'career_success_score'
ID_COL   = 'student_id'
TEXT_COL = 'mentor_feedback_text'

SKILL_COLS = [
    'coding_score', 'problem_solving_score', 'data_structures_score',
    'sql_score', 'machine_learning_score', 'backend_score',
    'frontend_score', 'cloud_score', 'devops_score'
]
SOCIAL_COLS = ['communication_score', 'teamwork_score', 'leadership_score', 'presentation_score']
CAT_COLS    = ['department', 'target_role', 'university_tier', 'hobby', 'preferred_social_media_platform']
MISSING_COLS= [
    'english_exam_score', 'internship_duration_months', 'portfolio_score',
    'github_avg_stars', 'open_source_contribution_count',
    'linkedin_profile_score', 'hr_interview_score'
]

NEG_WORDS = [
    'gelistirmeli', 'yetersiz', 'zayif', 'eksik',
    'calismasi gerekiyor', 'gelisime ihtiyac', 'iyilestirmeli',
    'gelistirilmesi gereken', 'uzerinde calismasi',
    'sinirli', 'kisitli', 'faydali olacaktir',
    'ihtiyac duyuyor', 'ogrenmesi gerekiyor', 'eksiklikleri var',
    'daha fazla deneyim', 'daha fazla calisma', 'yetersiz kalmakta',
    'geliştirilmeli', 'zayıf', 'sınırlı', 'kısıtlı',
    'üzerinde çalışması', 'çalışması gerekiyor',
    'gelişime ihtiyaç', 'iyileştirmeli',
    'geliştirilmesi gereken', 'ihtiyaç duyuyor',
    'öğrenmesi gerekiyor', 'eksiklikleri var',
    'yetersiz kalmakta', 'faydalı olacaktır'
]
POS_WORDS = [
    'mukemmel', 'basarili', 'guclu', 'etkileyici', 'dikkat cekici',
    'yetkin', 'uzman', 'olaganustu', 'ustun', 'parlak',
    'ileri duzey', 'vizyoner', 'tutkulu', 'kariyer potansiyeli',
    'aranan aday', 'potansiyeli buyuk', 'guclu teknik', 'hizli ogrenen',
    'sonuc odakli', 'kapsamli',
    'mükemmel', 'başarılı', 'güçlü',
    'etkileyici', 'dikkat çekici', 'yetkin', 'uzman',
    'olağanüstü', 'üstün', 'parlak',
    'ileri düzey', 'vizyoner', 'tutkulu', 'kariyer potansiyeli',
    'aranan aday', 'potansiyeli büyük', 'güçlü teknik',
    'hızlı öğrenen', 'sonuç odaklı', 'kapsamlı'
]

y = train[TARGET].copy()

## 3. NLP — Word, Char ve Combined Ridge Meta-Özellikleri

Üç farklı TF-IDF vektörleştirici ile Ridge regresyon meta-özellikleri üretilir:

| Vektörleştirici | Konfigürasyon | Ridge alpha | Amaç |
|-----------------|--------------|-------------|------|
| Word TF-IDF | unigram–trigram, max 5.000 özellik | 5.0 | Kelime düzeyinde anlam |
| Char TF-IDF | 3-gram–5-gram, max 10.000 özellik | 3.0 | Morfoloji ve yazım varyantları |
| Combined | Word + Char hstack | 4.0 | İkisini birleştiren genel sinyal |

Her Ridge modeli 5-fold OOF stratejisi ile eğitilir.
Üretilen OOF tahminleri doğrudan ağaç modellerine meta-özellik olarak aktarılacaktır.


In [4]:
def preprocess_turkish(text):
    if not isinstance(text, str):
        return ""
    # Python str.lower() İ->i ve I->ı dönüşümlerini hatalı işler.
    # Bu eşleme Türkçe büyük harfleri doğru küçük harfe çevirir.
    mapping = {'I': 'ı', 'İ': 'i', 'Ş': 'ş', 'Ğ': 'ğ', 'Ç': 'ç', 'Ö': 'ö', 'Ü': 'ü'}
    for k, v in mapping.items():
        text = text.replace(k, v)
    return text.lower()

all_texts = pd.concat([train[TEXT_COL], test[TEXT_COL]], ignore_index=True).apply(preprocess_turkish)

word_vec = TfidfVectorizer(max_features=5000, ngram_range=(1, 3), min_df=3, sublinear_tf=True)
X_word = word_vec.fit_transform(all_texts)
X_train_word = X_word[:len(train)]
X_test_word = X_word[len(train):]

char_vec = TfidfVectorizer(max_features=10000, analyzer='char_wb', ngram_range=(3, 5), min_df=3, sublinear_tf=True)
X_char = char_vec.fit_transform(all_texts)
X_train_char = X_char[:len(train)]
X_test_char = X_char[len(train):]

from scipy.sparse import hstack
X_comb = hstack([X_word, X_char]).tocsr()
X_train_comb = X_comb[:len(train)]
X_test_comb = X_comb[len(train):]

kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
oof_ridge_word = np.zeros(len(train))
oof_ridge_char = np.zeros(len(train))
oof_ridge_comb = np.zeros(len(train))
test_ridge_word = np.zeros(len(test))
test_ridge_char = np.zeros(len(test))
test_ridge_comb = np.zeros(len(test))

print('Word Ridge fit ediliyor...')
for tr_i, val_i in kf.split(X_train_word):
    ridge_w = Ridge(alpha=5.0, random_state=SEED)
    ridge_w.fit(X_train_word[tr_i], y.iloc[tr_i])
    oof_ridge_word[val_i] = ridge_w.predict(X_train_word[val_i])
    test_ridge_word += ridge_w.predict(X_test_word) / N_FOLDS

print('Char Ridge fit ediliyor...')
for tr_i, val_i in kf.split(X_train_char):
    ridge_c = Ridge(alpha=3.0, random_state=SEED)
    ridge_c.fit(X_train_char[tr_i], y.iloc[tr_i])
    oof_ridge_char[val_i] = ridge_c.predict(X_train_char[val_i])
    test_ridge_char += ridge_c.predict(X_test_char) / N_FOLDS

print('Combined Ridge fit ediliyor...')
for tr_i, val_i in kf.split(X_train_comb):
    ridge_comb = Ridge(alpha=4.0, random_state=SEED)
    ridge_comb.fit(X_train_comb[tr_i], y.iloc[tr_i])
    oof_ridge_comb[val_i] = ridge_comb.predict(X_train_comb[val_i])
    test_ridge_comb += ridge_comb.predict(X_test_comb) / N_FOLDS

Word Ridge fit ediliyor...
Char Ridge fit ediliyor...
Combined Ridge fit ediliyor...


## 4. Derin Öğrenme — BERTurk Sequence Regressor (GPU)

GPU ortamı tespit edilirse BERTurk fine-tuning pipeline'ı çalışır.

Pipeline adımları:
1. Hedef değişken fold bazında z-score normalleştirilir (train fold ortalaması/std ile).
2. `FeedbackDataset` sınıfı tokenizasyon ve padding işlemini yönetir.
3. Her fold için model sıfırdan yüklenir, 3 epoch eğitilir, validasyon ve test tahminleri alınır.
4. Tahminler ters dönüşümle (z-score -> orijinal ölçek) geri çevrilir.
5. GPU belleği her fold sonunda temizlenir.

Her fold için modelin sıfırdan yüklenmesi, fold'lar arası bilgi sızıntısını (data leakage) önler.
CPU fallback modunda bu hücre atlanır; `oof_bert` ve `test_bert` sıfır dizi olarak kalır.


In [5]:
oof_bert = np.zeros(len(train))
test_bert = np.zeros(len(test))

if USE_GPU_BERT:
    print('🚀 BERTurk Fine-Tuning Başlıyor (GPU)...')
    MODEL_NAME = 'dbmdz/bert-base-turkish-cased'
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    
    class FeedbackDataset(Dataset):
        def __init__(self, texts, scores=None, max_len=128):
            self.texts = texts
            self.scores = scores
            self.max_len = max_len
        def __len__(self):
            return len(self.texts)
        def __getitem__(self, idx):
            text = str(self.texts[idx])
            inputs = tokenizer(
                text, add_special_tokens=True, max_length=self.max_len,
                padding='max_length', truncation=True, return_token_type_ids=False,
                return_attention_mask=True, return_tensors='pt'
            )
            item = {
                'input_ids': inputs['input_ids'].flatten(),
                'attention_mask': inputs['attention_mask'].flatten()
            }
            if self.scores is not None:
                item['score'] = torch.tensor(self.scores[idx], dtype=torch.float)
            return item
            
    # Her fold için model sıfırdan yüklenir; fold'lar arası bilgi sızıntısı bu şekilde önlenir.
    for fold, (tr_i, val_i) in enumerate(kf.split(train)):
        print(f'  BERT Fold {fold+1}/{N_FOLDS}')
        y_tr = y.iloc[tr_i].values
        y_mean = y_tr.mean()
        y_std = y_tr.std()
        
        y_tr_scaled = (y_tr - y_mean) / y_std
        
        train_ds = FeedbackDataset(train[TEXT_COL].iloc[tr_i].values, y_tr_scaled)
        val_ds   = FeedbackDataset(train[TEXT_COL].iloc[val_i].values)
        test_ds  = FeedbackDataset(test[TEXT_COL].values)
        
        train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
        val_loader   = DataLoader(val_ds, batch_size=32, shuffle=False)
        test_loader  = DataLoader(test_ds, batch_size=32, shuffle=False)
        
        model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=1)
        model = model.to('cuda')
        
        optimizer = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
        scheduler = get_linear_schedule_with_warmup(
            optimizer, num_warmup_steps=0, num_training_steps=len(train_loader)*3
        )
        
        # 3 epoch: daha az -> eksik öğrenme; daha fazla -> GPU'da aşırı öğrenme riski.
        for epoch in range(3):
            model.train()
            for batch in train_loader:
                optimizer.zero_grad()
                input_ids = batch['input_ids'].to('cuda')
                attention_mask = batch['attention_mask'].to('cuda')
                scores = batch['score'].to('cuda')
                
                outputs = model(input_ids, attention_mask=attention_mask, labels=scores)
                loss = outputs.loss
                loss.backward()
                optimizer.step()
                scheduler.step()
                
        # Validasyon tahmini z-score'dan orijinal ölçeğe geri dönüştürülür.
        model.eval()
        val_preds = []
        with torch.no_grad():
            for batch in val_loader:
                input_ids = batch['input_ids'].to('cuda')
                attention_mask = batch['attention_mask'].to('cuda')
                outputs = model(input_ids, attention_mask=attention_mask)
                val_preds.extend(outputs.logits.flatten().cpu().numpy())
        oof_bert[val_i] = np.array(val_preds) * y_std + y_mean
        
        # Test tahmini her fold'da birikim toplamına eklenir; döngü sonunda fold sayısına bölünür.
        test_preds = []
        with torch.no_grad():
            for batch in test_loader:
                input_ids = batch['input_ids'].to('cuda')
                attention_mask = batch['attention_mask'].to('cuda')
                outputs = model(input_ids, attention_mask=attention_mask)
                test_preds.extend(outputs.logits.flatten().cpu().numpy())
        test_bert += (np.array(test_preds) * y_std + y_mean) / N_FOLDS
        
        # GPU belleği boşaltılır. Kaggle T4/P100 sınırlı VRAM'inde bir sonraki fold için yer açılır.
        del model, optimizer, scheduler
        torch.cuda.empty_cache()
    
    print(f'✅ BERTurk OOF MSE: {mean_squared_error(y, oof_bert):.4f}')
else:
    print('⚠️ GPU bulunamadı veya PyTorch yüklü değil. BERTurk fine-tuning aşaması atlanıyor. (Local CPU Fallback Modu)')

🚀 BERTurk Fine-Tuning Başlıyor (GPU)...


config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/60.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

  BERT Fold 1/5


...

BertForSequenceClassification LOAD REPORT from: dbmdz/bert-base-turkish-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  BERT Fold 5/5


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: dbmdz/bert-base-turkish-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✅ BERTurk OOF MSE: 131.9547


## 5. Tabular Özellik Mühendisliği ve Target Encoding

### `build_base_features` fonksiyonu

Train ve test kümeleri birleşik `raw` DataFrame üzerinde işlenir.
İstatistikler tüm veri üzerinden hesaplanır; ancak doldurma değerleri (medyan vb.)
yalnızca train setinden türetilir.

İşlem sırası:
1. **Eksik veri göstergeleri** — `MISSING_COLS` içindeki 7 sütun için binary bayrak oluşturulur.
2. **Akıllı sıfır doldurma** — `internship_count == 0` olan satırlar için `internship_duration_months` sıfır ile doldurulur.
3. **Medyan doldurma** — kalan sayısal sütunlar train medyanı ile doldurulur.
4. **Aykırı değer kırpma** — skewed 6 sütunda %1–%99 aralığına clip uygulanır.
5. **Teknik beceri istatistikleri** — ortalama, max, min, std, en iyi 3'ün ortalaması.
6. **Bileşik indeksler** — `combined_master`, `weighted_exp`, `portfolio_strength`, `github_strength`, `academic_str`, `career_readiness`.
7. **Etkileşim terimleri** — yüksek korelasyonlu özellik çiftleri çarpımı.
8. **Mezuniyet yılı farkı** (`years_to_grad`) ve kategorik bin.
9. **Üniversite kademesi** sayısallaştırma ve `tier_x_cgpa` etkileşimi.


In [6]:
def build_base_features(tr, te):
    n_tr = len(tr)
    raw = pd.concat([
        tr.drop(columns=[TARGET, ID_COL, TEXT_COL], errors='ignore'),
        te.drop(columns=[ID_COL, TEXT_COL], errors='ignore')
    ], ignore_index=True)
    
    for c in MISSING_COLS:
        raw[f'{c}_miss'] = np.concatenate([
            tr[c].isnull().astype(np.int8).values,
            te[c].isnull().astype(np.int8).values
        ])
    raw.loc[raw['internship_count'] == 0, 'internship_duration_months'] = \
        raw.loc[raw['internship_count'] == 0, 'internship_duration_months'].fillna(0)
    for c in raw.select_dtypes(include=np.number).columns:
        med = tr[c].median() if c in tr.columns else raw[c].median()
        raw[c] = raw[c].fillna(med)
    for c in CAT_COLS:
        raw[c] = raw[c].fillna('unknown')
    for c in ['github_avg_stars','github_repo_count','applications_sent',
               'interviews_attended','hackathon_awards','open_source_contribution_count']:
        lo, hi = raw[c].quantile([0.01, 0.99])
        raw[c] = raw[c].clip(lower=lo, upper=hi)

    raw['avg_tech']   = raw[SKILL_COLS].mean(axis=1)
    raw['max_tech']   = raw[SKILL_COLS].max(axis=1)
    raw['min_tech']   = raw[SKILL_COLS].min(axis=1)
    raw['std_tech']   = raw[SKILL_COLS].std(axis=1)
    raw['top3_tech']  = raw[SKILL_COLS].apply(lambda r: r.nlargest(3).mean(), axis=1)
    raw['avg_social'] = raw[SOCIAL_COLS].mean(axis=1)
    raw['max_social'] = raw[SOCIAL_COLS].max(axis=1)
    raw['combined_master'] = (
        raw['project_quality_score']     * 0.35 +
        raw['technical_interview_score'] * 0.20 +
        raw['avg_tech']                  * 0.25 +
        raw['avg_social']                * 0.10 +
        raw['portfolio_score']           * 0.10
    )
    raw['project_x_tech']   = raw['project_quality_score'] * raw['avg_tech']
    raw['interview_x_tech'] = raw['technical_interview_score'] * raw['avg_tech']
    raw['social_x_project'] = raw['avg_social'] * raw['project_quality_score']
    raw['cgpa_x_project']   = raw['cgpa'] * raw['project_quality_score']
    raw['weighted_exp'] = (
        raw['internship_count']           * 3.0 +
        raw['internship_duration_months'] * 0.5 +
        raw['real_client_project_count']  * 2.0 +
        raw['freelance_project_count']    * 1.5 +
        raw['hackathon_count']            * 1.0
    )
    raw['hackathon_wr']       = raw['hackathon_awards'] / (raw['hackathon_count'] + 1)
    raw['portfolio_strength'] = (raw['portfolio_score']*0.4 +
                                 raw['linkedin_profile_score']*0.3 +
                                 raw['cv_quality_score']*0.3)
    raw['github_strength']    = (raw['github_repo_count']*0.5 +
                                 raw['github_avg_stars']*2.0 +
                                 raw['open_source_contribution_count']*1.5)
    raw['academic_str']       = (raw['cgpa']*10 + raw['english_exam_score']*0.1 -
                                 raw['failed_courses_count']*2 + raw['attendance_rate']*0.1)
    raw['interview_success']  = (raw['technical_interview_score'] + raw['hr_interview_score']) / 2
    raw['interview_rate']     = raw['interviews_attended'] / (raw['applications_sent'] + 1)
    raw['career_readiness']   = raw['certification_count']*2 + raw['bootcamp_count']*3

    raw['problem_solving_x_cloud'] = raw['problem_solving_score'] * raw['cloud_score']
    raw['coding_x_problem_solving'] = raw['coding_score'] * raw['problem_solving_score']
    raw['project_x_interview'] = raw['project_quality_score'] * raw['technical_interview_score']
    raw['cgpa_x_tech'] = raw['cgpa'] * raw['avg_tech']

    raw['years_to_grad'] = raw['graduation_year'] - raw['application_year']
    tr_ytg = tr['graduation_year'] - tr['application_year']
    ytg_lo, ytg_hi = tr_ytg.quantile(0.01), tr_ytg.quantile(0.99)
    raw['years_to_grad'] = raw['years_to_grad'].clip(lower=ytg_lo, upper=ytg_hi)
    raw['years_to_grad_bin'] = pd.cut(
        raw['years_to_grad'], bins=[-999, 3, 4, 5, 6, 999], labels=[0, 1, 2, 3, 4]
    ).astype(float)
    for c in ['application_year', 'graduation_year']:
        if c in raw.columns:
            lo = tr[c].min() if c in tr.columns else raw[c].min()
            hi = tr[c].max() if c in tr.columns else raw[c].max()
            raw[c] = raw[c].clip(lower=lo, upper=hi)

    raw['tier_num']    = raw['university_tier'].map({'Tier 1':4,'Tier 2':3,'Tier 3':2,'Tier 4':1,'unknown':2}).fillna(2)
    raw['tier_x_cgpa'] = raw['tier_num'] * raw['cgpa']
    return raw.iloc[:n_tr].reset_index(drop=True), raw.iloc[n_tr:].reset_index(drop=True)

X_train, X_test = build_base_features(train, test)

In [7]:
# Mentor geri bildirim metninden duygu sinyali hesaplanır.
# sentiment_ratio = pozitif / (negatif + 1); sıfır bölme koruması için +1 eklenir.
neg_counts = all_texts.apply(lambda t: sum(1 for w in NEG_WORDS if w in t))
pos_counts = all_texts.apply(lambda t: sum(1 for w in POS_WORDS if w in t))
X_train['neg_s'] = neg_counts.values[:len(train)]
X_train['pos_s'] = pos_counts.values[:len(train)]
X_train['net_s'] = X_train['pos_s'] - X_train['neg_s']
X_train['sentiment_ratio'] = X_train['pos_s'] / (X_train['neg_s'] + 1)
X_test['neg_s']  = neg_counts.values[len(train):]
X_test['pos_s']  = pos_counts.values[len(train):]
X_test['net_s']  = X_test['pos_s'] - X_test['neg_s']
X_test['sentiment_ratio'] = X_test['pos_s'] / (X_test['neg_s'] + 1)

tfidf_svd = TfidfVectorizer(max_features=600, ngram_range=(1, 2), min_df=3, sublinear_tf=True)
svd = TruncatedSVD(n_components=15, random_state=SEED)
X_svd_all = svd.fit_transform(tfidf_svd.fit_transform(all_texts))
for i in range(15):
    if abs(pd.Series(X_svd_all[:len(train), i]).corr(y)) > 0.04:
        X_train[f'svd_{i}'] = X_svd_all[:len(train), i]
        X_test[f'svd_{i}']  = X_svd_all[len(train):, i]

# Bolum 3'te uretilen OOF Ridge tahminleri agac modellerine meta-ozellik olarak eklenir.
X_train['text_ridge_word'] = oof_ridge_word
X_train['text_ridge_char'] = oof_ridge_char
X_train['text_ridge_comb'] = oof_ridge_comb
X_test['text_ridge_word'] = test_ridge_word
X_test['text_ridge_char'] = test_ridge_char
X_test['text_ridge_comb'] = test_ridge_comb

# GPU modunda BERTurk OOF tahmini meta-ozellik olarak eklenir.
# CPU modunda bu sutun sifir kalir; agac modelleri uzerinde notr etkisi vardir.
X_train['bert_pred'] = oof_bert
X_test['bert_pred']  = test_bert

In [8]:
# OOF Bayesian (Smooth) Target Encoding
#
# Her kategorik sutun icin hedef ortalama, global ortalamaya dogru duzlestirilir.
# Formul: (n * kategori_ort + smoothing * global_ort) / (n + smoothing)
# Az ornekli kategorilerde asiri uyumu onler; OOF dongusu train setinde veri sizintisini engeller.
def add_oof_target_encoding(train_df, test_df, cat_cols, target, n_splits=5, seed=42, smoothing=10):
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    global_mean = target.mean()
    for col in cat_cols:
        train_df[f'{col}_te'] = global_mean
        test_df[f'{col}_te']  = global_mean
        
    for tr_i, val_i in kf.split(train_df):
        tr_x = train_df.iloc[tr_i]
        val_x = train_df.iloc[val_i]
        tr_y = target.iloc[tr_i]
        
        for col in cat_cols:
            col_series_tr = tr_x[col]
            if isinstance(col_series_tr, pd.DataFrame):
                col_series_tr = col_series_tr.iloc[:, 0]
            col_series_val = val_x[col]
            if isinstance(col_series_val, pd.DataFrame):
                col_series_val = col_series_val.iloc[:, 0]
                
            stats = tr_y.groupby(col_series_tr).agg(['count', 'mean'])
            stats['smooth'] = (stats['count'] * stats['mean'] + smoothing * global_mean) / (stats['count'] + smoothing)
            mapped_vals = col_series_val.map(stats['smooth']).fillna(global_mean).values
            
            loc = train_df.columns.get_loc(f'{col}_te')
            if isinstance(loc, np.ndarray) or isinstance(loc, slice):
                idx = np.where(train_df.columns == f'{col}_te')[0][0]
            else:
                idx = loc
            train_df.iloc[val_i, idx] = mapped_vals
            
    for col in cat_cols:
        col_series_train = train_df[col]
        if isinstance(col_series_train, pd.DataFrame):
            col_series_train = col_series_train.iloc[:, 0]
        col_series_test = test_df[col]
        if isinstance(col_series_test, pd.DataFrame):
            col_series_test = col_series_test.iloc[:, 0]
            
        stats = target.groupby(col_series_train).agg(['count', 'mean'])
        stats['smooth'] = (stats['count'] * stats['mean'] + smoothing * global_mean) / (stats['count'] + smoothing)
        mapped_vals_test = col_series_test.map(stats['smooth']).fillna(global_mean).values
        
        loc = test_df.columns.get_loc(f'{col}_te')
        if isinstance(loc, np.ndarray) or isinstance(loc, slice):
            idx = np.where(test_df.columns == f'{col}_te')[0][0]
        else:
            idx = loc
        test_df.iloc[:, idx] = mapped_vals_test
        
    return train_df, test_df

X_train, X_test = add_oof_target_encoding(X_train, X_test, CAT_COLS, y, smoothing=10)

le = LabelEncoder()
for c in CAT_COLS:
    le.fit(pd.concat([train[c], test[c]]).astype(str))
    X_train[f'{c}_le'] = le.transform(train[c].astype(str))
    X_test[f'{c}_le']  = le.transform(test[c].astype(str))

X_train_final = X_train.select_dtypes(exclude=['object'])
X_test_final  = X_test.select_dtypes(exclude=['object'])
print(f'Nihai Değişken Sayısı: {X_train_final.shape[1]}')

Nihai Değişken Sayısı: 101


## 6. Model Eğitimi ve OOF Tahminler

`oof_train_predict` — LightGBM, XGBoost ve CatBoost için ortak 5-fold OOF eğitim döngüsü.

Her model aynı fold bölünmesini kullanır (SEED=42); karşılaştırmalar tutarlı olur.
Early stopping (60 round) ile gereksiz ağaç sayısı engellenir.

Her modelin kategorik sütunları farklı biçimde tükettiğine dikkat edin:
- LightGBM: sütun adı listesi (`categorical_feature`)
- CatBoost: sütun indeksi listesi (`cat_features`)
- XGBoost: label encoding uygulanmış sayısal değerler


In [9]:
CAT_FEAT_NAMES = [f'{c}_le' for c in CAT_COLS]
CAT_FEAT_IDX   = [X_train_final.columns.get_loc(c) for c in CAT_FEAT_NAMES]

def oof_train_predict(model, X_tr, y_tr, X_te, name, is_lgb=False, is_xgb=False, is_cb=False):
    kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    oof       = np.zeros(len(X_tr))
    test_pred = np.zeros(len(X_te))
    
    for fold, (tr_i, val_i) in enumerate(kf.split(X_tr)):
        Xf_tr, Xf_val = X_tr.iloc[tr_i], X_tr.iloc[val_i]
        yf_tr, yf_val = y_tr.iloc[tr_i], y_tr.iloc[val_i]
        if is_lgb:
            model.fit(Xf_tr, yf_tr, eval_set=[(Xf_val, yf_val)],
                      categorical_feature=CAT_FEAT_NAMES,
                      callbacks=[lgb.early_stopping(60, verbose=False)])
        elif is_xgb:
            model.fit(Xf_tr, yf_tr, eval_set=[(Xf_val, yf_val)], verbose=False)
        elif is_cb:
            model.fit(Xf_tr, yf_tr, eval_set=(Xf_val, yf_val),
                      cat_features=CAT_FEAT_IDX, verbose=False)
        oof[val_i]   = model.predict(Xf_val)
        test_pred   += model.predict(X_te) / N_FOLDS
    oof_mse = mean_squared_error(y_tr, oof)
    print(f'  [{name}] OOF MSE = {oof_mse:.4f}')
    return oof, test_pred, oof_mse

oof_col = {}; test_col = {}; mse_res = {}
KEY_MAP = {'LightGBM':'lgb', 'XGBoost':'xgb', 'CatBoost':'cb'}

In [10]:
print('🔵 LightGBM Eğitiliyor...')
lgb_model = lgb.LGBMRegressor(
    n_estimators=3000, learning_rate=0.02, num_leaves=45, max_depth=6,
    min_child_samples=45, subsample=0.80, subsample_freq=1,
    colsample_bytree=0.75, reg_alpha=1.5, reg_lambda=4.0,
    random_state=SEED, n_jobs=-1, verbose=-1
)
oof_col['lgb'], test_col['lgb'], mse_res['LightGBM'] = \
    oof_train_predict(lgb_model, X_train_final, y, X_test_final, 'LightGBM', is_lgb=True)

🔵 LightGBM Eğitiliyor...
  [LightGBM] OOF MSE = 76.1574


In [11]:
print('🟠 XGBoost Eğitiliyor...')
xgb_model = xgb.XGBRegressor(
    n_estimators=3000, learning_rate=0.02, max_depth=5, min_child_weight=6,
    subsample=0.80, colsample_bytree=0.75, reg_alpha=1.5, reg_lambda=4.0,
    gamma=0.15, early_stopping_rounds=60, eval_metric='rmse',
    tree_method='hist', random_state=SEED, n_jobs=-1
)
oof_col['xgb'], test_col['xgb'], mse_res['XGBoost'] = \
    oof_train_predict(xgb_model, X_train_final, y, X_test_final, 'XGBoost', is_xgb=True)

🟠 XGBoost Eğitiliyor...
  [XGBoost] OOF MSE = 76.3405


In [12]:
print('🟡 CatBoost Eğitiliyor...')
cb_model = cb.CatBoostRegressor(
    iterations=3000, learning_rate=0.02, depth=5, l2_leaf_reg=6,
    subsample=0.80, early_stopping_rounds=60, random_seed=SEED, verbose=0
)
oof_col['cb'], test_col['cb'], mse_res['CatBoost'] = \
    oof_train_predict(cb_model, X_train_final, y, X_test_final, 'CatBoost', is_cb=True)

🟡 CatBoost Eğitiliyor...
  [CatBoost] OOF MSE = 75.3062


## 7. Ensemble Blending ve Nihai Sonuçlar

Üç modelin OOF tahminleri ağırlıklı ortalama ile birleştirilir.

| Mod | LightGBM | XGBoost | CatBoost | Gerekçe |
|-----|----------|---------|----------|---------|
| GPU (BERTurk aktif) | 0.20 | 0.05 | 0.75 | BERTurk meta-özellikleri CatBoost'u güçlendirir |
| CPU Fallback | 0.30 | 0.07 | 0.63 | v10 optimizasyonundan elde edilen blend ağırlıkları |

Beklenen test MSE hesaplaması: OOF hatası yıl bazında ağırlıklandırılır.
Test setindeki uygulama yılı dağılımı ağırlık vektörü olarak kullanılır.
Bu tahmin, public leaderboard skoru ile OOF arasındaki gap'i önceden ölçmeye yarar.


In [13]:
# Agirlikli ensemble blend. GPU/CPU moduna gore farkli katsayilar kullanilir.
if USE_GPU_BERT:
    # BERTurk meta-ozellikleri CatBoost'a derin semantik sinyal katar; bu nedenle CatBoost agirligi yuksek tutulur.
    w_lgb, w_xgb, w_cb = 0.20, 0.05, 0.75
else:
    # BERTurk devre disi; v10 optimizasyonundan elde edilen blend agirliklari uygulanir.
    w_lgb, w_xgb, w_cb = 0.30, 0.07, 0.63

blend_oof  = w_lgb * oof_col['lgb'] + w_xgb * oof_col['xgb'] + w_cb * oof_col['cb']
blend_test = w_lgb * test_col['lgb'] + w_xgb * test_col['xgb'] + w_cb * test_col['cb']
blend_mse  = mean_squared_error(y, blend_oof)

print('='*70)
print('  v11 NİHAİ MODEL SONUÇLARI')
print('='*70)
for name, mse_v in sorted(mse_res.items(), key=lambda x: x[1]):
    print(f'  {name:<28} OOF MSE = {mse_v:.4f}')
print(f'  {"Weighted Blend (v11)":<28} OOF MSE = {blend_mse:.4f}')

# Test setinin yil dagilimi agirlik vektoru olarak kullanilarak beklenen test MSE hesaplanir.
# Bu tahmin, public leaderboard skoru ile OOF arasindaki gap'i onceden olcer.
test_counts = test['application_year'].value_counts(normalize=True).to_dict()
train['blend_oof'] = blend_oof
train['se_blend'] = (train[TARGET] - train['blend_oof'])**2
mse_per_year = train.groupby('application_year')['se_blend'].mean().to_dict()
weighted_blend_mse = sum(mse_per_year[yr] * test_counts[yr] for yr in test_counts.keys())
print(f'  {"Weighted Blend (Expected Test)":<28} MSE = {weighted_blend_mse:.4f}')
print('='*70)

  v11 NİHAİ MODEL SONUÇLARI
  CatBoost                     OOF MSE = 75.3062
  LightGBM                     OOF MSE = 76.1574
  XGBoost                      OOF MSE = 76.3405
  Weighted Blend (v11)         OOF MSE = 75.1001
  Weighted Blend (Expected Test) MSE = 85.2937


In [14]:
# Blend tahminleri [0, 100] araligina kirpilir (hedef degisken tanimi geregi).
# Submission CSV dosyasi olusturulur.
final_clipped = np.clip(blend_test, 0, 100)
submission = pd.DataFrame({'student_id': test[ID_COL], 'career_success_score': final_clipped})
submission.to_csv('submission.csv', index=False)
print('🎉 submission.csv (v11) başarıyla kaydedildi!')

🎉 submission.csv (v11) başarıyla kaydedildi!
